In [1]:
from coniferest.pineforest import PineForest
from coniferest.aadforest import AADForest
from coniferest.isoforest import IsolationForest
from coniferest.session import Session
from coniferest.session.callback import TerminateAfter, viewer_decision_callback
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
from coniferest.aadforest import AADForest
#import onnxruntime as rt

import sys
sys.path.insert(0, '../..')

from sn_clf.scripts.utils import load_features, plot_config

plot_config()
np.random.seed(42)

In [2]:
#oids, features = load_features('../../dr23-features/sid_snad_clf_r_100.dat', '../../dr23-features/feature_sn_hard.dat')
field = 626
feature_set = 'sn_hard'
oids, features = load_features(f'../../dr23-features/collected_by_field/dr23_oid_{field}.dat', 
                                f'../../dr23-features/collected_by_field_sn_hard/dr23_feature_{field}.dat')

# _, rb_features = load_features(f'../../dr23-features/collected_by_field/dr23_oid_{field}.dat', 
#                                f'../../dr23-features/collected_by_field_rb/dr23_feature_{field}.dat')

#_, rb_features = load_features(f'../../dr23-features/sid_snad_clf_r_100.dat', 
#                               f'../../dr23-features/feature_rb.dat')


#features = np.hstack([features, rb_features[:,-1].reshape(-1,1)])

rs = 1
train_data = pd.read_csv('../data/train_data_big.csv')
budget = 30

feature_names = f'../../dr23-features/feature_{feature_set}.name'
with open(feature_names) as f:
    names = f.read().split()
    
print(oids.shape)
print(features.shape)

(310392,)
(310392, 48)


In [4]:
names

['amplitude_magn_r',
 'ln1p_anderson_darling_normal_magn_r',
 'beyond_1_std_magn_r',
 'beyond_2_std_magn_r',
 'cusum_magn_r',
 'lg_inter_percentile_range_2_magn_r',
 'lg_inter_percentile_range_10_magn_r',
 'lg_inter_percentile_range_25_magn_r',
 'kurtosis_magn_r',
 'arcsinh_linear_fit_slope_magn_r',
 'lg_linear_fit_slope_sigma_magn_r',
 'ln1p_linear_fit_reduced_chi2_magn_r',
 'arcsinh_linear_trend_magn_r',
 'lg_linear_trend_sigma_magn_r',
 'lg_linear_trend_noise_magn_r',
 'mean_magn_r',
 'median_absolute_deviation_magn_r',
 'period_0_magn_r',
 'period_s_to_n_0_magn_r',
 'period_1_magn_r',
 'period_s_to_n_1_magn_r',
 'period_2_magn_r',
 'period_s_to_n_2_magn_r',
 'period_3_magn_r',
 'period_s_to_n_3_magn_r',
 'period_4_magn_r',
 'period_s_to_n_4_magn_r',
 'periodogram_amplitude_magn_r',
 'periodogram_beyond_2_std_magn_r',
 'periodogram_beyond_3_std_magn_r',
 'periodogram_standard_deviation_magn_r',
 'ln1p_chi2_magn_r',
 'arcsinh_skew_magn_r',
 'standard_deviation_magn_r',
 'stetson_K_ma

In [3]:
# remove bts SNe from anomaly detection dataset
mask = ~np.isin(oids, train_data[train_data['is_sn'] == 1]['oid'])
oids = oids[mask]
features = features[mask]
features.shape

(310387, 48)

In [4]:
sn_oid = np.array(train_data[train_data['is_sn'] == 1]['oid'], dtype=np.uint64)
sn_priors = np.array(train_data[train_data['is_sn'] == 1][names], dtype=np.float32)

N = sn_oid.shape[0]
M = 10
a = np.arange(0, N)
rand_ind = np.random.choice(a, M, replace=False)

sn_oid = sn_oid[rand_ind]
sn_priors = sn_priors[rand_ind]

assert (sn_oid == pd.read_csv('../pineforest_output/init_rs1_field795.csv')['oid'].iloc[:10]).all(), "Priors aren't the same!"

In [5]:
feat_concat = np.vstack([sn_priors, features])
oid_concat = np.hstack([sn_oid, oids])
print(feat_concat.shape)

(310397, 48)


In [6]:
labels = dict(zip(np.arange(0, sn_priors.shape[0]),
                  np.ones(sn_priors.shape[0])*(-1)))

In [4]:
s = Session(features, oids, #feat_concat, oid_concat,
            #known_labels=labels,
            on_decision_callbacks= [TerminateAfter(budget)],
            decision_callback=viewer_decision_callback,
            model=PineForest(random_seed=rs)) #, n_trees=20, n_spare_trees=80

In [5]:
t = time.monotonic()
s.run()
t = (time.monotonic() - t) / 60
print(f'Session took {(t / 60):.0f}h. {t:.0f}m.')

Check https://ztf.snad.space/view/626202100005115 for details
Is 626202100005115 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  n


Check https://ztf.snad.space/view/626216200003094 for details
Is 626216200003094 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  n


Check https://ztf.snad.space/view/626205400000129 for details
Is 626205400000129 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

...

Check https://ztf.snad.space/view/626209200005257 for details
Is 626209200005257 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  n


Check https://ztf.snad.space/view/626205200002584 for details
Is 626205200002584 an anomaly? ([A]nomaly / yes, [R]egular / no, [U]nknown):

  n


Session took 0h. 10m.


In [6]:
is_anomaly = np.asarray(list(s.known_labels.values()))
is_anomaly[is_anomaly == 1] = np.full(len(is_anomaly[is_anomaly == 1]), 0)
is_anomaly[is_anomaly == -1] = np.full(len(is_anomaly[is_anomaly == -1]), 1)
checked_oids = oids[list(s.known_labels.keys())] #oid_concat
table = pd.DataFrame(data={'oid': checked_oids, 'is_anomaly': is_anomaly})

In [7]:
table['is_anomaly'].sum()

np.int64(0)

In [8]:
table.to_csv(f'../pineforest_output/aug_sn_no_priors_rs{rs}_field{field}.csv', index=False)

# top 30 objects by sn score

In [12]:
table_aug = pd.read_csv(f'../pineforest_output/aug_sn_rs{rs}_field{field}.csv')
table_init = pd.read_csv(f'../pineforest_output/init_rs{rs}_field{field}.csv')
table = pd.concat([table_aug.iloc[10:], table_init.iloc[10:]])

In [13]:
sn_score = features[:, -1]
ind = np.argsort(sn_score)[::-1][:30]
ans = []
for i in ind:
    if oids[i] in table['oid'].tolist():
        ans.append(table[table['oid'] == oids[i]]['is_anomaly'].iloc[0])
        continue
        
    url = f'https://ztf.snad.space/view/{oids[i]}'
    print(url)
    a = int(input())
    ans.append(a)
np.sum(ans)

https://ztf.snad.space/view/626201400018214


 0


https://ztf.snad.space/view/626210100000208


 0


https://ztf.snad.space/view/626212200016660


...

https://ztf.snad.space/view/626203300034682


 0


https://ztf.snad.space/view/626213400017594


 0


np.float64(1.0)

In [14]:
table = pd.DataFrame(data=[[oids[ind][i], ans[i], sn_score[ind][i]] for i in range(30)], columns=['oid', 'is_sn', 'sn_score'])
table.to_csv(f'../top30_snscore/field{field}.csv', index=False)